# 03b — Advanced Feature Engineering

---

This notebook extends the initial feature dataset with more advanced pre-match features.

## Objectives

The goal of this notebook is to improve the quality of the predictive dataset by adding features that are commonly useful in football match prediction:

- ELO ratings,
- home/away split rolling features,
- rest days,
- cumulative season-to-date team statistics.

## Why this step matters

The previous feature engineering notebook created a first layer of rolling features based on recent performance.

In this notebook, we make the dataset more realistic and more informative by capturing:

- long-term team strength through ELO,
- venue-specific form through home/away rolling features,
- fatigue and scheduling effects through rest days,
- and broader season context through cumulative season-to-date statistics.

As before, all features must be computed using only information that was available before the current match.

---

## 1. Imports and setup

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from src.utils import ensure_directories
from src.feature_engineering import add_match_outcome_points
from src.advanced_features import (
    build_long_team_match_table,
    add_rest_days,
    add_overall_rolling_features,
    add_home_away_split_rolling_features,
    add_cumulative_season_features,
    compute_elo_ratings,
    split_home_away_feature_tables,
    merge_advanced_match_features,
    add_feature_differences,
)

---

## 2. Load the base match dataset

We start from the clean match-level dataset created earlier.

This dataset already contains:
- one row per match,
- final score,
- xG values,
- target variables,
- points for home and away teams.

In [2]:
base_path = INTERIM_DATA_DIR / "base_matches.parquet"
csv_fallback_path = INTERIM_DATA_DIR / "base_matches.csv"

if base_path.exists():
    try:
        df_matches = pd.read_parquet(base_path)
        print(f"Loaded parquet base dataset from: {base_path}")
    except Exception as exc:
        if not csv_fallback_path.exists():
            raise
        print(f"Parquet read failed ({exc}). Falling back to CSV: {csv_fallback_path}")
        df_matches = pd.read_csv(csv_fallback_path)
else:
    df_matches = pd.read_csv(csv_fallback_path)
    print(f"Loaded CSV base dataset from: {csv_fallback_path}")

df_matches["date"] = pd.to_datetime(df_matches["date"], errors="coerce")

print("Base matches shape:", df_matches.shape)
display(df_matches.head())


Loaded parquet base dataset from: C:\Users\cerve\Desktop\DP\match_prediction\data\interim\base_matches.parquet
Base matches shape: (882, 36)


,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,home_team_code,away_team_code,...,away_ppda,home_deep_completions,away_deep_completions,target_1x2,home_win,draw,away_win,round,week,matchday
0,3,2023,23065,2023-08-18 18:30:00,123,117,Werder Bremen,Bayern Munich,WER,BAY,...,19.714286,2,16,A,0,0,1,Bundesliga,1,1
1,3,2023,23066,2023-08-19 13:30:00,119,136,Bayer Leverkusen,RasenBallsport Leipzig,LEV,RBL,...,12.105263,5,4,H,1,0,0,Bundesliga,1,1
2,3,2023,23067,2023-08-19 13:30:00,131,280,Wolfsburg,FC Heidenheim,WOL,HEI,...,16.578947,8,7,H,1,0,0,Bundesliga,1,1
3,3,2023,23068,2023-08-19 13:30:00,120,135,Hoffenheim,Freiburg,HOF,FRE,...,21.727273,8,7,A,0,0,1,Bundesliga,1,1
4,3,2023,23069,2023-08-19 13:30:00,121,130,Augsburg,Borussia M.Gladbach,AUG,BMG,...,19.5,12,7,D,0,1,0,Bundesliga,1,1


---

## 3. Add match points

The long-format team-match table requires points won by the home and away teams.

If these columns are not already present in the saved base dataset, we recreate them here before building the long-format table.

In [3]:
from src.feature_engineering import add_match_outcome_points
df_matches = add_match_outcome_points(df_matches)
display(df_matches[[
    "date", "home_team", "away_team",
    "home_goals", "away_goals",
    "home_points", "away_points"
]].head())

,date,home_team,away_team,home_goals,away_goals,home_points,away_points
0,2023-08-18 18:30:00,Werder Bremen,Bayern Munich,0,4,0.0,3.0
1,2023-08-19 13:30:00,Bayer Leverkusen,RasenBallsport Leipzig,3,2,3.0,0.0
2,2023-08-19 13:30:00,Wolfsburg,FC Heidenheim,2,0,3.0,0.0
3,2023-08-19 13:30:00,Hoffenheim,Freiburg,1,2,0.0,3.0
4,2023-08-19 13:30:00,Augsburg,Borussia M.Gladbach,4,4,1.0,1.0


---

## 4. Build the long-format team-match table

Before creating advanced team-level features, we convert the match-level dataset into a long format where each match produces two rows:
- one row for the home team,
- one row for the away team.

This representation is required for rolling features, rest days, and cumulative season statistics.

In [4]:
df_matches = add_match_outcome_points(df_matches)

df_long = build_long_team_match_table(df_matches)

print("Long-format shape:", df_long.shape)
display(df_long.head(10))

Long-format shape: (1764, 20)


,game_id,date,season_id,league_id,team,opponent,goals_for,goals_against,xg_for,xg_against,np_xg_for,np_xg_against,expected_points,ppda,deep_completions,points,is_home,goal_diff,xg_diff,np_xg_diff
0,23069,2023-08-19 13:30:00,2023,3,Augsburg,Borussia M.Gladbach,4,4,2.53482,1.91849,1.77704,1.16071,1.8311,11.176471,12,1.0,1,0,0.61633,0.61633
1,23082,2023-08-27 15:30:00,2023,3,Augsburg,Bayern Munich,1,3,0.726241,2.82296,0.726241,2.06518,0.201,16.545455,6,0.0,0,-2,-2.096719,-1.338939
2,23087,2023-09-02 13:30:00,2023,3,Augsburg,Bochum,2,2,1.88106,1.86494,1.88106,1.86494,1.3996,18.0,3,1.0,1,0,0.01612,0.01612
3,23093,2023-09-16 13:30:00,2023,3,Augsburg,RasenBallsport Leipzig,0,3,1.34046,1.46279,1.34046,1.46279,1.2679,17.9,7,0.0,0,-3,-0.12233,-0.12233
4,23102,2023-09-23 13:30:00,2023,3,Augsburg,Mainz 05,2,1,1.04683,0.487325,1.04683,0.487325,1.9351,13.363636,2,3.0,1,1,0.559505,0.559505
5,23118,2023-10-01 15:30:00,2023,3,Augsburg,Freiburg,0,2,1.00081,1.2721,1.00081,0.51432,1.0663,11.565217,2,0.0,0,-2,-0.27129,0.48649
6,23122,2023-10-07 13:30:00,2023,3,Augsburg,Darmstadt,1,2,1.18473,1.96153,1.18473,1.20376,0.8073,9.64,12,0.0,1,-1,-0.7768,-0.01903
7,23136,2023-10-22 15:30:00,2023,3,Augsburg,FC Heidenheim,5,2,2.82353,1.99327,2.06575,1.99327,1.9913,22.8125,4,3.0,0,3,0.83026,0.07248
8,23144,2023-10-28 13:30:00,2023,3,Augsburg,Wolfsburg,3,2,1.59374,1.25516,1.59374,0.497379,1.6435,16.736842,5,3.0,1,1,0.33858,1.096361
9,23151,2023-11-04 14:30:00,2023,3,Augsburg,FC Cologne,1,1,2.54685,2.22618,2.54685,2.22618,1.6295,16.0,4,1.0,0,0,0.32067,0.32067


## 5. Add rest days

One simple but often useful contextual variable is the number of days since a team's previous match.

This feature can capture scheduling pressure, fatigue, and unequal preparation time between opponents.

For the first match of a team in the dataset, rest days will be missing, which is expected.

In [5]:
df_long = add_rest_days(df_long)

display(
    df_long[[
        "date", "team", "opponent", "is_home",
        "prev_date", "rest_days"
    ]].head(10)
)

,date,team,opponent,is_home,prev_date,rest_days
0,2023-08-19 13:30:00,Augsburg,Borussia M.Gladbach,1,NaT,NaN
1,2023-08-27 15:30:00,Augsburg,Bayern Munich,0,2023-08-19 13:30:00,8.0
2,2023-09-02 13:30:00,Augsburg,Bochum,1,2023-08-27 15:30:00,5.0
3,2023-09-16 13:30:00,Augsburg,RasenBallsport Leipzig,0,2023-09-02 13:30:00,14.0
4,2023-09-23 13:30:00,Augsburg,Mainz 05,1,2023-09-16 13:30:00,7.0
5,2023-10-01 15:30:00,Augsburg,Freiburg,0,2023-09-23 13:30:00,8.0
6,2023-10-07 13:30:00,Augsburg,Darmstadt,1,2023-10-01 15:30:00,5.0
7,2023-10-22 15:30:00,Augsburg,FC Heidenheim,0,2023-10-07 13:30:00,15.0
8,2023-10-28 13:30:00,Augsburg,Wolfsburg,1,2023-10-22 15:30:00,5.0
9,2023-11-04 14:30:00,Augsburg,FC Cologne,0,2023-10-28 13:30:00,7.0


## 6. Add overall recent-form features

We first create pre-match features based on all previous matches, regardless of venue.

Instead of relying only on the overlapping windows `last 3` and `last 5`, we now mix three different views of form:
- a short rolling window (`last 2`) for immediate form,
- a broader rolling window (`last 8`) for medium-term stability,
- an exponentially weighted moving average (`ewm span 5`) that gives more weight to recent matches without using a hard cut-off.

All of these are computed using `shift(1)`, so the current match is never included in its own features.


In [6]:
# The helper now creates short-horizon, medium-horizon, and recency-weighted form features.
df_long = add_overall_rolling_features(df_long, windows=[2, 8], ewm_spans=[5])

display(df_long.head(10))


,game_id,date,season_id,league_id,team,opponent,goals_for,goals_against,xg_for,xg_against,...,points_ewm_span_5_overall,goal_diff_avg_last_2_overall,goal_diff_avg_last_8_overall,goal_diff_ewm_span_5_overall,xg_diff_avg_last_2_overall,xg_diff_avg_last_8_overall,xg_diff_ewm_span_5_overall,np_xg_diff_avg_last_2_overall,np_xg_diff_avg_last_8_overall,np_xg_diff_ewm_span_5_overall
0,23069,2023-08-19 13:30:00,2023,3,Augsburg,Borussia M.Gladbach,4,4,2.53482,1.91849,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,23082,2023-08-27 15:30:00,2023,3,Augsburg,Bayern Munich,1,3,0.726241,2.82296,...,1.000000,0.0,0.000000,0.000000,0.616330,0.616330,0.616330,0.616330,0.616330,0.616330
2,23087,2023-09-02 13:30:00,2023,3,Augsburg,Bochum,2,2,1.88106,1.86494,...,0.666667,-1.0,-1.000000,-0.666667,-0.740195,-0.740195,-0.288020,-0.361304,-0.361304,-0.035426
3,23093,2023-09-16 13:30:00,2023,3,Augsburg,RasenBallsport Leipzig,0,3,1.34046,1.46279,...,0.777778,-1.0,-0.666667,-0.444444,-1.040300,-0.488090,-0.186640,-0.661409,-0.235496,-0.018244
4,23102,2023-09-23 13:30:00,2023,3,Augsburg,Mainz 05,2,1,1.04683,0.487325,...,0.518519,-1.5,-1.250000,-1.296296,-0.053105,-0.396650,-0.165203,-0.053105,-0.207205,-0.052939
5,23118,2023-10-01 15:30:00,2023,3,Augsburg,Freiburg,0,2,1.00081,1.2721,...,1.345679,-1.0,-0.800000,-0.530864,0.218587,-0.205419,0.076366,0.218587,-0.053863,0.151209
6,23122,2023-10-07 13:30:00,2023,3,Augsburg,Darmstadt,1,2,1.18473,1.96153,...,0.897119,-0.5,-1.000000,-1.020576,0.144107,-0.216397,-0.039519,0.522998,0.036196,0.262969
7,23136,2023-10-22 15:30:00,2023,3,Augsburg,FC Heidenheim,5,2,2.82353,1.99327,...,0.598080,-1.5,-1.000000,-1.013717,-0.524045,-0.296455,-0.285279,0.233730,0.028307,0.168969
8,23144,2023-10-28 13:30:00,2023,3,Augsburg,Wolfsburg,3,2,1.59374,1.25516,...,1.398720,1.0,-0.500000,0.324188,0.026730,-0.155616,0.086567,0.026725,0.033828,0.136806
9,23151,2023-11-04 14:30:00,2023,3,Augsburg,FC Cologne,1,1,2.54685,2.22618,...,1.932480,2.0,-0.375000,0.549459,0.584420,-0.190334,0.170571,0.584420,0.093832,0.456658


---

## 7. Add venue-specific recent-form features

Venue still matters, so we create the same family of form features separately for home and away contexts.

This gives the model three complementary venue-aware signals:
- immediate venue form (`last 2`),
- medium-term venue form (`last 8`),
- and a smooth recency-weighted venue trend (`ewm span 5`).

These venue-specific features often describe team strength more realistically than raw overall averages alone.

---


In [7]:
# Venue-specific helpers mirror the overall setup: short, medium, and recency-weighted form.
df_long = add_home_away_split_rolling_features(df_long, windows=[2, 8], ewm_spans=[5])

display(df_long.head(10))


,game_id,date,season_id,league_id,team,opponent,goals_for,goals_against,xg_for,xg_against,...,points_ewm_span_5_venue,goal_diff_avg_last_2_venue,goal_diff_avg_last_8_venue,goal_diff_ewm_span_5_venue,xg_diff_avg_last_2_venue,xg_diff_avg_last_8_venue,xg_diff_ewm_span_5_venue,np_xg_diff_avg_last_2_venue,np_xg_diff_avg_last_8_venue,np_xg_diff_ewm_span_5_venue
0,23069,2023-08-19 13:30:00,2023,3,Augsburg,Borussia M.Gladbach,4,4,2.53482,1.91849,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,23082,2023-08-27 15:30:00,2023,3,Augsburg,Bayern Munich,1,3,0.726241,2.82296,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,23087,2023-09-02 13:30:00,2023,3,Augsburg,Bochum,2,2,1.88106,1.86494,...,1.000000,0.0,0.000000,0.000000,0.616330,0.616330,0.616330,0.616330,0.616330,0.616330
3,23093,2023-09-16 13:30:00,2023,3,Augsburg,RasenBallsport Leipzig,0,3,1.34046,1.46279,...,0.000000,-2.0,-2.000000,-2.000000,-2.096719,-2.096719,-2.096719,-1.338939,-1.338939,-1.338939
4,23102,2023-09-23 13:30:00,2023,3,Augsburg,Mainz 05,2,1,1.04683,0.487325,...,1.000000,0.0,0.000000,0.000000,0.316225,0.316225,0.416260,0.316225,0.316225,0.416260
5,23118,2023-10-01 15:30:00,2023,3,Augsburg,Freiburg,0,2,1.00081,1.2721,...,0.000000,-2.5,-2.500000,-2.333333,-1.109525,-1.109525,-1.438589,-0.730634,-0.730634,-0.933403
6,23122,2023-10-07 13:30:00,2023,3,Augsburg,Darmstadt,1,2,1.18473,1.96153,...,1.666667,0.5,0.333333,0.333333,0.287812,0.397318,0.464008,0.287812,0.397318,0.464008
7,23136,2023-10-22 15:30:00,2023,3,Augsburg,FC Heidenheim,5,2,2.82353,1.99327,...,0.000000,-2.5,-2.333333,-2.222222,-0.196810,-0.830113,-1.049490,0.182080,-0.324926,-0.460105
8,23144,2023-10-28 13:30:00,2023,3,Augsburg,Wolfsburg,3,2,1.59374,1.25516,...,1.111111,0.0,0.000000,-0.111111,-0.108648,0.103789,0.050406,0.270238,0.293231,0.302996
9,23151,2023-11-04 14:30:00,2023,3,Augsburg,FC Cologne,1,1,2.54685,2.22618,...,1.000000,0.5,-1.000000,-0.481481,0.279485,-0.415020,-0.422906,0.279485,-0.225575,-0.282577


---

## 8. Add cumulative season-to-date features

Short rolling windows capture recent form, but they do not capture the broader season context.

To complement them, we now create cumulative season-to-date statistics based on matches played earlier in the same season.

Examples include:
- average goals scored so far in the season,
- average xG conceded so far in the season,
- average points per match so far.

These features can represent medium-term team quality within the current season.

---

In [8]:
df_long = add_cumulative_season_features(df_long)

display(df_long.head(10))

,game_id,date,season_id,league_id,team,opponent,goals_for,goals_against,xg_for,xg_against,...,goals_for_cum_avg_before,goals_against_cum_avg_before,xg_for_cum_avg_before,xg_against_cum_avg_before,np_xg_for_cum_avg_before,np_xg_against_cum_avg_before,expected_points_cum_avg_before,ppda_cum_avg_before,deep_completions_cum_avg_before,points_cum_avg_before
0,23069,2023-08-19 13:30:00,2023,3,Augsburg,Borussia M.Gladbach,4,4,2.53482,1.91849,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,23082,2023-08-27 15:30:00,2023,3,Augsburg,Bayern Munich,1,3,0.726241,2.82296,...,4.000000,4.000000,2.53482,1.91849,1.77704,1.16071,1.8311,11.176471,12.0,1.000000
2,23087,2023-09-02 13:30:00,2023,3,Augsburg,Bochum,2,2,1.88106,1.86494,...,2.500000,3.500000,1.63053,2.370725,1.251641,1.612945,1.01605,13.860963,9.0,0.500000
3,23093,2023-09-16 13:30:00,2023,3,Augsburg,RasenBallsport Leipzig,0,3,1.34046,1.46279,...,2.333333,3.000000,1.71404,2.20213,1.461447,1.696943,1.1439,15.240642,7.0,0.666667
4,23102,2023-09-23 13:30:00,2023,3,Augsburg,Mainz 05,2,1,1.04683,0.487325,...,1.750000,3.000000,1.620645,2.017295,1.4312,1.638405,1.1749,15.905481,7.0,0.500000
5,23118,2023-10-01 15:30:00,2023,3,Augsburg,Freiburg,0,2,1.00081,1.2721,...,1.800000,2.600000,1.505882,1.711301,1.354326,1.408189,1.32694,15.397112,6.0,1.000000
6,23122,2023-10-07 13:30:00,2023,3,Augsburg,Darmstadt,1,2,1.18473,1.96153,...,1.500000,2.500000,1.421703,1.638101,1.295407,1.259211,1.2835,14.758463,5.333333,0.833333
7,23136,2023-10-22 15:30:00,2023,3,Augsburg,FC Heidenheim,5,2,2.82353,1.99327,...,1.428571,2.428571,1.38785,1.684305,1.279596,1.251289,1.215471,14.027254,6.285714,0.714286
8,23144,2023-10-28 13:30:00,2023,3,Augsburg,Wolfsburg,3,2,1.59374,1.25516,...,1.875000,2.375000,1.56731,1.722926,1.377865,1.344037,1.31245,15.12541,6.0,1.000000
9,23151,2023-11-04 14:30:00,2023,3,Augsburg,FC Cologne,1,1,2.54685,2.22618,...,2.000000,2.333333,1.570247,1.670952,1.401851,1.249964,1.349233,15.304458,5.888889,1.222222


## 9. Inspect advanced features for one team

Before merging anything back into the match-level table, it is useful to check whether the new features behave logically for one specific team.

This is a sanity check for:
- chronological order,
- rolling feature construction,
- rest days,
- cumulative statistics.

In [9]:
team_example = df_long["team"].iloc[0]

team_view_cols = [
    "date",
    "team",
    "opponent",
    "is_home",
    "goals_for",
    "goals_against",
    "points",
    "rest_days",
    "goals_for_avg_last_2_overall",
    "goals_for_avg_last_8_overall",
    "goals_for_ewm_span_5_overall",
    "goals_for_avg_last_2_venue",
    "goals_for_ewm_span_5_venue",
    "points_cum_avg_before",
]

display(df_long[df_long["team"] == team_example][team_view_cols].head(12))


,date,team,opponent,is_home,goals_for,goals_against,points,rest_days,goals_for_avg_last_2_overall,goals_for_avg_last_8_overall,goals_for_ewm_span_5_overall,goals_for_avg_last_2_venue,goals_for_ewm_span_5_venue,points_cum_avg_before
0,2023-08-19 13:30:00,Augsburg,Borussia M.Gladbach,1,4,4,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-08-27 15:30:00,Augsburg,Bayern Munich,0,1,3,0.0,8.0,4.0,4.000000,4.000000,NaN,NaN,1.000000
2,2023-09-02 13:30:00,Augsburg,Bochum,1,2,2,1.0,5.0,2.5,2.500000,3.000000,4.0,4.000000,0.500000
3,2023-09-16 13:30:00,Augsburg,RasenBallsport Leipzig,0,0,3,0.0,14.0,1.5,2.333333,2.666667,1.0,1.000000,0.666667
4,2023-09-23 13:30:00,Augsburg,Mainz 05,1,2,1,3.0,7.0,1.0,1.750000,1.777778,3.0,3.333333,0.500000
5,2023-10-01 15:30:00,Augsburg,Freiburg,0,0,2,0.0,8.0,1.0,1.800000,1.851852,0.5,0.666667,1.000000
6,2023-10-07 13:30:00,Augsburg,Darmstadt,1,1,2,0.0,5.0,1.0,1.500000,1.234568,2.0,2.888889,0.833333
7,2023-10-22 15:30:00,Augsburg,FC Heidenheim,0,5,2,3.0,15.0,0.5,1.428571,1.156379,0.0,0.444444,0.714286
8,2023-10-28 13:30:00,Augsburg,Wolfsburg,1,3,2,3.0,5.0,3.0,1.875000,2.437586,1.5,2.259259,1.000000
9,2023-11-04 14:30:00,Augsburg,FC Cologne,0,1,1,1.0,7.0,4.0,1.750000,2.625057,2.5,1.962963,1.222222


## 10. Compute ELO ratings on the match-level table

We now return to the match-level dataset and compute sequential ELO ratings.

### Why ELO?

ELO is a compact way to represent team strength based on historical results.  
It is especially useful because it evolves over time and provides a dynamic estimate of relative quality.

### Important detail

For prediction, we use **pre-match ELO** values:
- `home_elo_pre`
- `away_elo_pre`

These are the ratings that were known before the match started.

The post-match ratings are also computed internally, but they are not used as predictive inputs.

In [10]:
df_matches_elo = compute_elo_ratings(
    df_matches,
    k=20.0,
    home_adv=80.0,
    base=1500.0,
)

display(
    df_matches_elo[[
        "date", "home_team", "away_team",
        "home_elo_pre", "away_elo_pre", "elo_diff_pre"
    ]].head(10)
)

,date,home_team,away_team,home_elo_pre,away_elo_pre,elo_diff_pre
0,2023-08-18 18:30:00,Werder Bremen,Bayern Munich,1500.000000,1500.000000,0.000000
1,2023-08-19 13:30:00,Bayer Leverkusen,RasenBallsport Leipzig,1500.000000,1500.000000,0.000000
2,2023-08-19 13:30:00,Wolfsburg,FC Heidenheim,1500.000000,1500.000000,0.000000
3,2023-08-19 13:30:00,Hoffenheim,Freiburg,1500.000000,1500.000000,0.000000
4,2023-08-19 13:30:00,Augsburg,Borussia M.Gladbach,1500.000000,1500.000000,0.000000
5,2023-08-19 13:30:00,VfB Stuttgart,Bochum,1500.000000,1500.000000,0.000000
6,2023-08-19 16:30:00,Borussia Dortmund,FC Cologne,1500.000000,1500.000000,0.000000
7,2023-08-20 13:30:00,Union Berlin,Mainz 05,1500.000000,1500.000000,0.000000
8,2023-08-20 15:30:00,Eintracht Frankfurt,Darmstadt,1500.000000,1500.000000,0.000000
9,2023-08-25 18:30:00,RasenBallsport Leipzig,VfB Stuttgart,1492.262736,1507.737264,-15.474527


---

## 11. Split the long-format features into home and away tables

The advanced features currently live in the team-level long dataset.

To make them usable for modeling, we now split them into:
- home-side feature rows,
- away-side feature rows,

and then merge them back into one match-level table.

---

In [11]:
home_features, away_features = split_home_away_feature_tables(df_long)

print("Home features shape:", home_features.shape)
print("Away features shape:", away_features.shape)

Home features shape: (882, 92)
Away features shape: (882, 92)


---

## 12. Merge advanced team features back into match-level data

After this merge, each row will again represent one match, but now with a much richer set of pre-match predictors describing both teams.

---

In [12]:
df_advanced = merge_advanced_match_features(
    df_matches_elo,
    home_features,
    away_features,
)

print("Advanced feature dataset shape:", df_advanced.shape)
display(df_advanced.head())

Advanced feature dataset shape: (882, 221)


,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,home_team_code,away_team_code,...,away_goals_for_cum_avg_before,away_goals_against_cum_avg_before,away_xg_for_cum_avg_before,away_xg_against_cum_avg_before,away_np_xg_for_cum_avg_before,away_np_xg_against_cum_avg_before,away_expected_points_cum_avg_before,away_ppda_cum_avg_before,away_deep_completions_cum_avg_before,away_points_cum_avg_before
0,3,2023,23065,2023-08-18 18:30:00,123,117,Werder Bremen,Bayern Munich,WER,BAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,2023,23066,2023-08-19 13:30:00,119,136,Bayer Leverkusen,RasenBallsport Leipzig,LEV,RBL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2023,23067,2023-08-19 13:30:00,131,280,Wolfsburg,FC Heidenheim,WOL,HEI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2023,23068,2023-08-19 13:30:00,120,135,Hoffenheim,Freiburg,HOF,FRE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,2023,23069,2023-08-19 13:30:00,121,130,Augsburg,Borussia M.Gladbach,AUG,BMG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
(df_advanced.columns).tolist()

['league_id',
 'season_id',
 'game_id',
 'date',
 'home_team_id',
 'away_team_id',
 'home_team',
 'away_team',
 'home_team_code',
 'away_team_code',
 'home_goals',
 'away_goals',
 'home_xg',
 'away_xg',
 'is_result',
 'has_data',
 'url',
 'home_points',
 'away_points',
 'home_expected_points',
 'away_expected_points',
 'home_np_xg',
 'away_np_xg',
 'home_np_xg_difference',
 'away_np_xg_difference',
 'home_ppda',
 'away_ppda',
 'home_deep_completions',
 'away_deep_completions',
 'target_1x2',
 'home_win',
 'draw',
 'away_win',
 'round',
 'week',
 'matchday',
 'home_elo_pre',
 'away_elo_pre',
 'elo_diff_pre',
 'home_is_home',
 'home_rest_days',
 'home_goals_for_avg_last_2_overall',
 'home_goals_for_avg_last_8_overall',
 'home_goals_for_ewm_span_5_overall',
 'home_goals_against_avg_last_2_overall',
 'home_goals_against_avg_last_8_overall',
 'home_goals_against_ewm_span_5_overall',
 'home_xg_for_avg_last_2_overall',
 'home_xg_for_avg_last_8_overall',
 'home_xg_for_ewm_span_5_overall',
 'ho

---

## 13. Add difference features

In addition to absolute feature values for the home and away teams, it is often useful to model their difference directly.

For example:
- home ELO minus away ELO,
- home recent xG minus away recent xG,
- home rest days minus away rest days.

Instead of hard-coding every single rolling feature name, we now build the difference feature list programmatically. This keeps the notebook aligned with the current feature families, including the new short-window, medium-window, and recency-weighted form features.

---


In [14]:
difference_feature_names = [
    "rest_days",
    "matches_played_before",
    "goals_for_cum_avg_before",
    "goals_against_cum_avg_before",
    "xg_for_cum_avg_before",
    "xg_against_cum_avg_before",
    "np_xg_for_cum_avg_before",
    "np_xg_against_cum_avg_before",
    "expected_points_cum_avg_before",
    "ppda_cum_avg_before",
    "deep_completions_cum_avg_before",
    "points_cum_avg_before",
]

rolling_suffixes = (
    "_avg_last_2_overall",
    "_avg_last_8_overall",
    "_ewm_span_5_overall",
    "_avg_last_2_venue",
    "_avg_last_8_venue",
    "_ewm_span_5_venue",
)

for col in df_long.columns:
    if any(col.endswith(suffix) for suffix in rolling_suffixes):
        difference_feature_names.append(col)

difference_feature_names = sorted(set(difference_feature_names))

print("Number of difference features to create:", len(difference_feature_names))
print(difference_feature_names[:20])

df_advanced = add_feature_differences(df_advanced, difference_feature_names)
display(df_advanced.head())


Number of difference features to create: 90
['deep_completions_avg_last_2_overall', 'deep_completions_avg_last_2_venue', 'deep_completions_avg_last_8_overall', 'deep_completions_avg_last_8_venue', 'deep_completions_cum_avg_before', 'deep_completions_ewm_span_5_overall', 'deep_completions_ewm_span_5_venue', 'expected_points_avg_last_2_overall', 'expected_points_avg_last_2_venue', 'expected_points_avg_last_8_overall', 'expected_points_avg_last_8_venue', 'expected_points_cum_avg_before', 'expected_points_ewm_span_5_overall', 'expected_points_ewm_span_5_venue', 'goal_diff_avg_last_2_overall', 'goal_diff_avg_last_2_venue', 'goal_diff_avg_last_8_overall', 'goal_diff_avg_last_8_venue', 'goal_diff_ewm_span_5_overall', 'goal_diff_ewm_span_5_venue']


C:\Users\cerve\Desktop\DP\match_prediction\src\advanced_features.py:338: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"diff_{f}"] = df[f"home_{f}"] - df[f"away_{f}"]
C:\Users\cerve\Desktop\DP\match_prediction\src\advanced_features.py:338: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"diff_{f}"] = df[f"home_{f}"] - df[f"away_{f}"]
C:\Users\cerve\Desktop\DP\match_prediction\src\advanced_features.py:338: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

,league_id,season_id,game_id,date,home_team_id,away_team_id,home_team,away_team,home_team_code,away_team_code,...,diff_xg_diff_avg_last_8_venue,diff_xg_diff_ewm_span_5_overall,diff_xg_diff_ewm_span_5_venue,diff_xg_for_avg_last_2_overall,diff_xg_for_avg_last_2_venue,diff_xg_for_avg_last_8_overall,diff_xg_for_avg_last_8_venue,diff_xg_for_cum_avg_before,diff_xg_for_ewm_span_5_overall,diff_xg_for_ewm_span_5_venue
0,3,2023,23065,2023-08-18 18:30:00,123,117,Werder Bremen,Bayern Munich,WER,BAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,2023,23066,2023-08-19 13:30:00,119,136,Bayer Leverkusen,RasenBallsport Leipzig,LEV,RBL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2023,23067,2023-08-19 13:30:00,131,280,Wolfsburg,FC Heidenheim,WOL,HEI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2023,23068,2023-08-19 13:30:00,120,135,Hoffenheim,Freiburg,HOF,FRE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,2023,23069,2023-08-19 13:30:00,121,130,Augsburg,Borussia M.Gladbach,AUG,BMG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---

## 14. Inspect missing values

Some missing values are expected, especially:
- at the beginning of the dataset,
- at the beginning of each season,
- or when there is no previous home-only / away-only history.

This is normal for pre-match sports data.

At the modeling stage, we will decide how to handle these values consistently.

---

In [15]:
missing_share = df_advanced.isna().mean().sort_values(ascending=False)
display(missing_share.head(30).to_frame("missing_share"))

,missing_share
round,0.420635
matchday,0.148526
week,0.148526
away_ppda_cum_avg_before,0.031746
diff_ppda_cum_avg_before,0.031746
home_ppda_cum_avg_before,0.031746
away_expected_points_cum_avg_before,0.030612
away_xg_for_cum_avg_before,0.030612
away_np_xg_against_cum_avg_before,0.030612
away_np_xg_for_cum_avg_before,0.030612


---

## 15. Define the advanced model input dataset

We now select the columns that should form the advanced modeling dataset.

This includes:
- identifiers,
- targets,
- pre-match ELO,
- home and away advanced features,
- difference features.

---

In [16]:
# Standardize team-name columns
if "home_team" not in df_advanced.columns:
    if "home_team_x" in df_advanced.columns:
        df_advanced["home_team"] = df_advanced["home_team_x"]
    elif "home_team_y" in df_advanced.columns:
        df_advanced["home_team"] = df_advanced["home_team_y"]

if "away_team" not in df_advanced.columns:
    if "away_team_x" in df_advanced.columns:
        df_advanced["away_team"] = df_advanced["away_team_x"]
    elif "away_team_y" in df_advanced.columns:
        df_advanced["away_team"] = df_advanced["away_team_y"]

cols_to_drop = [
    "home_team_x", "away_team_x", "home_team_y", "away_team_y",
    "home_goals.1", "away_goals.1",
    "home_xg.1", "away_xg.1",
    "home_win.1", "away_win.1",
    "home_points_x", "away_points_x",
    "home_points_y", "away_points_y",
    "home_expected_points_x", "away_expected_points_x",
    "home_expected_points_y", "away_expected_points_y",
    "home_ppda_x", "away_ppda_x",
    "home_ppda_y", "away_ppda_y",
    "home_deep_completions_x", "away_deep_completions_x",
    "home_deep_completions_y", "away_deep_completions_y",
    "home_elo_pre.1", "away_elo_pre.1",
]

cols_to_drop = [col for col in cols_to_drop if col in df_advanced.columns]
df_advanced = df_advanced.drop(columns=cols_to_drop)

In [17]:
# Remove obvious duplicate columns that may appear from older cached runs or previous exports
duplicate_like_cols = [
    "home_team.1", "away_team.1",
    "home_goals.1", "away_goals.1",
    "home_xg.1", "away_xg.1",
    "home_win.1", "away_win.1",
    "home_elo_pre.1", "away_elo_pre.1",
]

duplicate_like_cols = [col for col in duplicate_like_cols if col in df_advanced.columns]
df_advanced = df_advanced.drop(columns=duplicate_like_cols)

print("Shape after duplicate cleanup:", df_advanced.shape)

Shape after duplicate cleanup: (882, 311)


In [18]:
id_target_columns = [
    "game_id",
    "date",
    "season_id",
    "league_id",
    "home_team_id",
    "away_team_id",
    "home_team",
    "away_team",
    "home_team_code",
    "away_team_code",
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "home_np_xg",
    "away_np_xg",
    "home_expected_points",
    "away_expected_points",
    "home_ppda",
    "away_ppda",
    "home_deep_completions",
    "away_deep_completions",
    "target_1x2",
    "home_win",
    "draw",
    "away_win",
    "home_elo_pre",
    "away_elo_pre",
    "elo_diff_pre",
]

advanced_feature_columns = [
    col for col in df_advanced.columns
    if (
        col.startswith("diff_")
        or (
            col.startswith("home_")
            and (
                "_avg_last_" in col
                or "_cum_avg_before" in col
                or col in ["home_is_home", "home_rest_days", "home_matches_played_before"]
            )
        )
        or (
            col.startswith("away_")
            and (
                "_avg_last_" in col
                or "_cum_avg_before" in col
                or col in ["away_is_home", "away_rest_days", "away_matches_played_before"]
            )
        )
    )
]

# Remove leakage-prone post-match columns if present
leakage_like_cols = {
    "home_home_points",
    "away_away_points",
    "home_target_1x2",
    "away_target_1x2",
    "home_goals_for",
    "home_goals_against",
    "away_goals_for",
    "away_goals_against",
}

advanced_feature_columns = [
    col for col in advanced_feature_columns
    if col not in leakage_like_cols
]

final_columns = id_target_columns + advanced_feature_columns
final_columns = [col for col in final_columns if col in df_advanced.columns]

df_model_advanced = df_advanced[final_columns].copy()

print("Advanced model input shape:", df_model_advanced.shape)
display(df_model_advanced.head())

Advanced model input shape: (882, 249)


,game_id,date,season_id,league_id,home_team_id,away_team_id,home_team,away_team,home_team_code,away_team_code,...,diff_xg_diff_avg_last_8_venue,diff_xg_diff_ewm_span_5_overall,diff_xg_diff_ewm_span_5_venue,diff_xg_for_avg_last_2_overall,diff_xg_for_avg_last_2_venue,diff_xg_for_avg_last_8_overall,diff_xg_for_avg_last_8_venue,diff_xg_for_cum_avg_before,diff_xg_for_ewm_span_5_overall,diff_xg_for_ewm_span_5_venue
0,23065,2023-08-18 18:30:00,2023,3,123,117,Werder Bremen,Bayern Munich,WER,BAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,23066,2023-08-19 13:30:00,2023,3,119,136,Bayer Leverkusen,RasenBallsport Leipzig,LEV,RBL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,23067,2023-08-19 13:30:00,2023,3,131,280,Wolfsburg,FC Heidenheim,WOL,HEI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,23068,2023-08-19 13:30:00,2023,3,120,135,Hoffenheim,Freiburg,HOF,FRE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,23069,2023-08-19 13:30:00,2023,3,121,130,Augsburg,Borussia M.Gladbach,AUG,BMG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---

## 16. Save the advanced feature dataset

This saved dataset will serve as the main input for the first modeling notebooks.


---

In [19]:
ensure_directories([PROCESSED_DATA_DIR])

output_path = PROCESSED_DATA_DIR / "match_features_advanced.parquet"

try:
    df_model_advanced.to_parquet(output_path, index=False)
except Exception:
    output_path = PROCESSED_DATA_DIR / "match_features_advanced.csv"
    df_model_advanced.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: C:\Users\cerve\Desktop\DP\match_prediction\data\processed\match_features_advanced.parquet


---

## 17. Summary

This notebook extended the original feature dataset with several important pre-match predictors.

### What was added

- rest days,
- overall rolling features,
- home/away split rolling features,
- cumulative season-to-date features,
- ELO ratings,
- home-away difference features.

### Why this matters

The dataset now captures:
- recent team form,
- venue-specific strength,
- accumulated seasonal performance,
- fatigue and scheduling effects,
- and dynamic team strength through ELO.

This gives us a much stronger basis for both:
- ML classification models,
- and the Poisson-based modeling branch.

### Next step

The next notebook should focus on baseline predictive modeling, beginning with:
- multinomial logistic regression,
- and one or two stronger non-linear models such as Random Forest or XGBoost.